# 16 · Terrain and water-level exploration in 3D

**Question:** How much of a dramatic terrain scene is data, and how much is display exaggeration?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Have Astra compare the 3D rendering with a cell-based area calculation and ask Codex for controls that preserve vertical units.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Surface, datum and screening plane · 30 minutes
**Learn:** construct a 3D surface from a raster, label a vertical datum, and compare a threshold area with a visual impression.
Plotly supplies an orbitable scientific surface rather than a tiled geographic basemap. Coordinates are local
east/north metres; elevations use an invented datum. Each of 40 × 40 cells represents 50 × 50 m, sampled at
its center. The surface interpolates between centers for display; area uses full cell counts.
**Predict:** does changing vertical exaggeration change the area below the water plane?
This threshold model omits drainage and hydraulic connectivity. It is not a flood forecast.


In [ ]:
from scenes import export_scene, scene_frame
pixel_m=50.; axis=(np.arange(40)+.5)*pixel_m
xx,yy=np.meshgrid(axis,axis)
elevation=1000+.003*xx+.002*yy-4*np.exp(-((xx-1100)/250)**2)
water_level=1001.
below=elevation<water_level
area_m2=int(below.sum())*pixel_m**2
levels=np.arange(998,1008.5,.5)
areas=np.array([np.count_nonzero(elevation<level)*pixel_m**2 for level in levels])
assert (np.diff(areas)>=0).all() and 0<=area_m2<=2000**2
assert elevation.shape==(40,40) and len(axis)*pixel_m==2000
print('Cells below plane:',int(below.sum()),'| screening area:',area_m2,'m²')
payload=dict(x=axis.tolist(),y=axis.tolist(),z=elevation.tolist(),pixel_m=pixel_m,
             synthetic=True,datum='invented local vertical reference',units='metres')
export_json('16_terrain.json',payload)


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
image=axes[0].imshow(elevation,origin='lower',extent=[0,2000,0,2000],cmap='terrain',vmin=995,vmax=1012)
axes[0].contour(xx,yy,elevation,levels=[water_level],colors=['#176caf'])
map_axes(axes[0],'Plan view and 1001 m contour'); fig.colorbar(image,ax=axes[0],label='Synthetic elevation (m)')
axes[1].plot(levels,areas/1e6,marker='o',color='#087f8c')
axes[1].set(xlabel='Water screening level (m)',ylabel='Area below plane (km²)',title='Cell-based sensitivity; not hydraulic extent')
plt.tight_layout(); plt.show()


## Use and share the interactive scene
The scene appears below in a trusted notebook output. If your notebook viewer blocks the iframe,
download the HTML from `exports/` and open it in a browser. All data are embedded, but the pinned
JavaScript/CSS renderer still downloads from a CDN; this is not fully offline. WebGL must be available.
The static plot and numerical outputs remain usable when the renderer cannot load.

Camera rotation and vertical exaggeration change the view, not the underlying data. Keep a plan view
beside a perspective view to detect occlusion. Download the notebook, scene and evidence JSON before
clearing browser storage. Do not label synthetic geometry as observed Gaborone buildings.


In [ ]:
document=export_scene('16_terrain_scene.html','terrain',payload)
display(scene_frame(document,'Synthetic terrain and water screening plane'))


## Astra map audit and a Codex surface task
**Astra:** send `16_terrain.json` and screenshots at 1× and 30× vertical exaggeration.
Ask: “How many cell centers lie below 1001 m, and what full-cell area does that represent? Why do the two
views imply different apparent slopes but the same area? Which evidence is missing for a real flood map?”
Grade count and area exactly against the cell calculation; review the scientific explanation separately.

**Codex:** “Add an elevation cross-section at a selected row to `content/scene_templates/terrain.html`.
Keep its y-axis in real metres regardless of 3D exaggeration. Mark the selected row on the scene and test
the cross-section against the corresponding exported array.”

**Acceptance:** water-level increases cannot reduce threshold area; exaggeration cannot change elevations
or cell counts; row zero is south; datum and units remain visible. Record the exact prompt and response before
using the reference calculation to evaluate it.

Reference: [Plotly JavaScript surface plots](https://plotly.com/javascript/3d-surface-plots/).
For georeferenced terrain tiles or a globe, MapLibre terrain or CesiumJS is a further extension requiring
documented elevation sources, licensing and vertical references.


## Extend with Codex or Astra
Replace the synthetic raster with a small licensed DEM chip after preprocessing CRS, nodata and vertical datum. Preserve a static plan view and hand-checked area calculation; separate hydraulic modeling from a water-plane visualization.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
